# 06 — Paper trading & stabilisation (Projet 4B)

Ce notebook **léger** s'appuie uniquement sur les snapshots JSON mensuels et sur `src/paper_trading.py` : il n'ajoute aucune stratégie et ne recalcule aucune formule financière.

> **Paper trading uniquement.** Aucun ordre n'est exécuté ; la décision reste humaine et fictive.

## 1. Initialisation

In [1]:
import sys
from pathlib import Path


def find_project_root(markers=("src", "config")) -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if all((candidate / m).is_dir() for m in markers):
            return candidate
    raise RuntimeError("Racine du projet introuvable.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
from src.paper_trading import (
    list_monthly_snapshots, load_monthly_snapshot,
    create_paper_entry_from_snapshot, save_paper_entry, load_paper_entry, load_paper_entries,
    compare_monthly_reports, update_manual_decision,
)

MONTHLY_DIR = PROJECT_ROOT / "reports" / "monthly"
PAPER_DIR = PROJECT_ROOT / "reports" / "paper_trading"
print("Racine :", PROJECT_ROOT)

Racine : C:\Users\degar\OneDrive - De Vinci Higher Education (DVHE)\Perso\Projet Investissement\personal-quant-lab


## 2. Lecture du dernier snapshot mensuel

In [2]:
snapshots = list_monthly_snapshots(MONTHLY_DIR)
assert snapshots, "Aucun snapshot mensuel : lance d'abord `python main.py monthly-report`."
latest_path = snapshots[-1]
latest = load_monthly_snapshot(latest_path)
print("Snapshots disponibles :", [p.name for p in snapshots])
print("Dernier :", latest_path.name, "| statut :", latest["metadata"]["pipeline_status"])

Snapshots disponibles : ['2026-07-report.json']
Dernier : 2026-07-report.json | statut : OK_WITH_WARNINGS


## 3. Création d'une entrée de paper trading (statut PENDING)

In [3]:
report_month = latest["metadata"]["report_month"]
entry_path = PAPER_DIR / f"{report_month}-paper-entry.json"

if entry_path.exists():
    # Rejeu idempotent : conserve toute annotation humaine existante.
    entry = load_paper_entry(entry_path)
else:
    markdown_path = latest_path.with_name(latest_path.name.replace("-report.json", "-report.md"))
    entry = create_paper_entry_from_snapshot(
        latest, latest_path, markdown_path if markdown_path.exists() else None
    )
    save_paper_entry(entry, entry_path)
    print("Entrée créée :", entry_path.name)
    print("Statut décision :", entry["manual_decision_status"], "| proposée :", entry["system_suggested_strategy"])

Entrée créée : 2026-07-paper-entry.json
Statut décision : PENDING | proposée : momentum_vt


## 4. Inspection des poids (allocations théoriques)

In [4]:
import pandas as pd
weights = {k: v for k, v in entry["target_weights"].items() if v is not None}
table = pd.DataFrame(weights).fillna(0.0)
print((table * 100).round(1).astype(str) + " %")

       benchmark momentum momentum_vt
CW8.PA   100.0 %    0.0 %       0.0 %
CASH       0.0 %    0.0 %      16.3 %
ESE.PA     0.0 %  100.0 %      83.7 %


## 5. Inspection des alertes

In [5]:
summary = entry["alerts_summary"]
print("Total alertes :", summary["total"], "| par niveau :", summary["counts"])
for a in latest["alerts"]:
    print(f"[{a['level']}] {a['code']} — {a['message']}")

Total alertes : 3 | par niveau : {'INFO': 2, 'WARNING': 1, 'CRITICAL': 0}
[INFO] DATA_NOT_REFRESHED — Rapport basé sur les données locales (pas de téléchargement).
[WARNING] ASSET_EXCLUDED — Actifs écartés faute d'historique : EWLD.PA.
[INFO] PARTIAL_MONTH — Le mois 2026-07 est incomplet et exclu du signal officiel.


## 6. Comparaison avec un snapshot précédent (si disponible)

In [6]:
if len(snapshots) >= 2:
    previous = load_monthly_snapshot(snapshots[-2])
    comparison = compare_monthly_reports(previous, latest)
    print(f"{comparison['previous_month']} -> {comparison['current_month']}")
else:
    # Un seul mois disponible : illustration de la fonction (aucun changement).
    comparison = compare_monthly_reports(latest, latest)
    print("Un seul snapshot disponible : comparaison illustrative (aucun changement).")

print("Momentum sélectionné :", comparison["momentum_selected_from"], "->", comparison["momentum_selected_to"])
print("Statut pipeline :", comparison["pipeline_status_from"], "->", comparison["pipeline_status_to"])
print("Nouvelles alertes :", comparison["new_alert_codes"] or "aucune")
json.dumps(comparison, allow_nan=False)  # vérifie la sérialisabilité
print("Comparaison sérialisable : OK")

Un seul snapshot disponible : comparaison illustrative (aucun changement).
Momentum sélectionné : ['ESE.PA'] -> ['ESE.PA']
Statut pipeline : OK_WITH_WARNINGS -> OK_WITH_WARNINGS
Nouvelles alertes : aucune
Comparaison sérialisable : OK


## 7. Exemple de décision fictive (non sauvegardée sur le journal officiel)

In [7]:
# Démonstration en mémoire : on n'écrase pas l'entrée officielle (reste PENDING).
demo = update_manual_decision(
    entry, "APPROVED_PAPER",
    strategy=entry["system_suggested_strategy"],
    comment="Décision fictive — paper trading uniquement",
    validated_at="2026-07-09T12:00:00",
)
print("Décision (démo) :", demo["manual_decision_status"], "| stratégie :", demo["selected_strategy"])
print("Poids retenus :", demo["manual_approved_weights"])
print("Entrée officielle inchangée :", entry["manual_decision_status"])

Décision (démo) : APPROVED_PAPER | stratégie : momentum_vt
Poids retenus : {'CW8.PA': 0.0, 'ESE.PA': 0.8367860721138229, 'CASH': 0.16321392788617706}
Entrée officielle inchangée : PENDING


## 8. Validation finale

In [8]:
# Journal relu depuis le disque
entries = load_paper_entries(PAPER_DIR)
assert entries, "Le journal doit contenir au moins une entrée."
official = [e for e in entries if e["report_month"] == entry["report_month"]][0]

# Au rejeu, l'entrée en mémoire est exactement celle déjà enregistrée :
# une annotation humaine n'est jamais écrasée par ce notebook.
assert official == entry
assert official["manual_decision_status"] in {"PENDING", "APPROVED_PAPER", "REJECTED", "SKIPPED", "NEEDS_REVIEW"}
raw = (PAPER_DIR / f"{entry['report_month']}-paper-entry.json").read_text(encoding="utf-8")
assert "NaN" not in raw and "Infinity" not in raw             # JSON standard
assert "EXECUTED" not in raw                                   # aucun ordre réel
assert official["source_snapshot_json"]                        # trace de la source
print("Tous les contrôles passent.")

Tous les contrôles passent.


In [9]:
print("Projet 4B — Paper trading & stabilisation validé techniquement.")
print("Aucune décision réelle ni aucun ordre n'a été exécuté.")

Projet 4B — Paper trading & stabilisation validé techniquement.
Aucune décision réelle ni aucun ordre n'a été exécuté.
